# NLP Assignment – 3: Chatbot using Transformers


---

### Objective
Build a conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.

### Model Used
- **microsoft/DialoGPT-medium** — A dialogue-focused GPT-2 based model, trained specifically for multi-turn conversations. It gives much better, more relevant responses compared to base GPT-2.

### Pipeline Flow
`User Input → Model Processing → Response Generation → Display Output → Loop Until Exit`

---
## Step 1: Install Required Libraries
We need the `transformers` library from Hugging Face and `torch` (PyTorch) as the backend.


In [2]:
# Install the Hugging Face transformers library and PyTorch
!pip install transformers torch --quiet

---
##Import Libraries


In [4]:
# Importing the required classes from Hugging Face transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

# PyTorch is used as the backend for model computation
import torch

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


---
## Load the Pre-trained Model


In [5]:
# Choosing DialoGPT-medium — trained specifically for dialogue/conversations
# It gives much more meaningful and contextual replies than base GPT-2
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"⏳ Loading model: {MODEL_NAME}")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set the model to evaluation mode
model.eval()

print("\n✅ Model loaded successfully! DialoGPT-medium is ready to chat.")

⏳ Loading model: microsoft/DialoGPT-medium


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


✅ Model loaded successfully! DialoGPT-medium is ready to chat.


---
## Define the Response Generation Function
This function takes the user's message and the conversation history, then returns the chatbot's reply.


In [13]:
def generate_response(user_message, conversation_history):

    # Step 1: Encode the user's message and add the end-of-string token
    # The EOS token tells the model where one turn ends and another begins
    new_input_ids = tokenizer.encode(
        user_message + tokenizer.eos_token,
        return_tensors="pt"
    )

    # Step 2: Append the new input to the existing conversation history
    # This gives the model context of the entire conversation so far
    if conversation_history is not None:
        bot_input_ids = torch.cat([conversation_history, new_input_ids], dim=-1)
    else:
        # First message — no history yet
        bot_input_ids = new_input_ids

    # Step 3: Generate the model's response
    # We keep the history to a max of 1000 tokens to avoid memory issues
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )

    # Step 4: Decode only the newly generated part (not the whole history)
    # We skip special tokens so the output is clean readable text
    reply = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    # Return the reply text and the updated conversation history
    return reply, chat_history_ids


print("✅ Response generation function is ready!")

✅ Response generation function is ready!


---
##Run the Chatbot



In [25]:
def run_chatbot():
    """
    Main function to run the conversational chatbot.
    Keeps the conversation going until the user types 'exit' or 'quit'.
    """

    # conversation_history stores all previous token IDs for context
    conversation_history = None

    # Greet the user when the chatbot starts
    print("=" * 55)
    print(" Chatbot using Transformers (DialoGPT-medium)")
    print("=" * 55)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 55)
    print("  (Type 'exit' or 'quit' to end the conversation)")
    print("-" * 55)

    # Keep the chatbot running in a loop until user decides to exit
    while True:

        user_input = input("\nYou: ").strip()

        if not user_input:
            print("Chatbot: Please type something so I can help you!")
            continue

        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: It was great talking to you! Goodbye!")
            print("=" * 55)
            break

        response, conversation_history = generate_response(user_input, conversation_history)

        # Display the chatbot's reply
        print(f"\nChatbot: {response}")


# Start the chatbot
run_chatbot()

 Chatbot using Transformers (DialoGPT-medium)
Chatbot: Hello! I am your AI assistant. How can I help you today?
-------------------------------------------------------
  (Type 'exit' or 'quit' to end the conversation)
-------------------------------------------------------

You: Hello

Chatbot: Hello. How's it going?

You: What is Artificail Intelligence?

Chatbot: Nice. I just don't understand how to say it.

You: who created python?

Chatbot: who made art?

You: thank you

Chatbot: I have no idea

You: Exit

Chatbot: It was great talking to you! Goodbye!
